In [9]:
!pip install -q transformers accelerate sentencepiece tqdm pandas torch


In [ ]:
import pandas as pd
test_df=pd.read_csv("datasets_llm_multitask_df.csv")
test_df

In [2]:
ps = test_df[test_df["task"] == "personalized_summarization"]
valid_queries = (
    ps["query_text_id"]
    .value_counts()
    .loc[lambda x: x >= 3]
    .index
)
ps_valid = ps[ps["query_text_id"].isin(valid_queries)]
selected = []
count = 0

for qid, group in ps_valid.groupby("query_text_id"):
    if count >= 100:
        break
    if count + len(group) <= 100:
        selected.append(group)
        count += len(group)
ps_100 = pd.concat(selected)


In [ ]:
ps_100

In [3]:
df_rest = test_df[~test_df.index.isin(ps_100.index)]


In [4]:
SEED = 42

# remove rows already used in ps_100
df_rest = test_df.drop(ps_100.index)

task_dfs = {}
task_dfs["personalized_summarization"] = ps_100

for task in ["next_item_prediction", "next_user_prediction", "dialogue_generation"]:
    subset = df_rest[df_rest["task"] == task]

    if len(subset) < 100:
        raise ValueError(f"{task} has only {len(subset)} rows")

    task_dfs[task] = subset.sample(n=100, random_state=SEED)


test_df = (
    pd.concat(task_dfs.values())
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)


In [ ]:
test_df

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

def llm_generate(prompt, max_new_tokens=256):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]["generated_text"]
    # If prompt echoed, strip it:
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()


In [6]:
import ast

def safe_parse(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x   # already a list

test_df["text_nodes"] = test_df["text_nodes"].apply(safe_parse)
test_df["interactions"] = test_df["interactions"].apply(safe_parse)


docs = pd.read_csv("datasets_documents.csv")
resp = pd.read_csv("response.csv")
dial = pd.read_csv("dialogue.csv")
summ = pd.read_csv("summaries (1).csv")


In [7]:
doc_map   = dict(zip(docs["NewsID"], docs["NewsBody"]))
resp_map  = dict(zip(resp["response_id"], resp["response_text"]))
dial_map  = dict(zip(dial["dialogue_id"], dial["dialogue_text"]))
summ_map  = dict(zip(summ["SummID"], summ["Summary"]))

In [8]:
def resolve_node(node_id):
    if node_id.startswith("d-"):
        return doc_map.get(node_id, "")
    if node_id.startswith("r-"):
        return resp_map.get(node_id, "")
    if node_id.startswith("dl-"):
        return dial_map.get(node_id, "")
    if node_id.startswith("s-"):
        return summ_map.get(node_id, "")
    return ""


In [9]:
def build_history(row):
    history_lines = []

    for node, act in zip(row["text_nodes"], row["interactions"]):
        text = resolve_node(node)

        if act == "pos":
            tag = "[USER INTEREST]"
        elif act == "neg":
            tag = "[USER DISINTEREST]"
        elif act in ["gen_res", "res_gen"]:
            tag = "[DIALOGUE TURN]"
        else:
            tag = "[INTERACTION]"

        history_lines.append(f"{tag}\n{text}")

    return "\n\n".join(history_lines)


In [ ]:
up_df = test_df[test_df["task"]=="next_user_prediction"].copy()

user_histories = {}

for uid, g in tqdm(up_df.groupby("userid")):
    texts = []
    for _, row in g.iterrows():
        texts.append(build_history(row))
    user_histories[uid] = "\n".join(texts[:300])


In [42]:
import re
import ast
from tqdm.auto import tqdm

def tokenize(s: str):
    return re.findall(r"[a-z]+", s.lower())

def ngrams(tokens, n):
    return {" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)}

def overlap_ngram_score(history: str, persona: str, n=5):
    ht = tokenize(history)
    pt = tokenize(persona)
    if len(pt) < n:
        return 0.0
    H = ngrams(ht, n)
    P = ngrams(pt, n)
    if not P:
        return 0.0
    return len(P & H) / max(1, len(P))

def has_obvious_leaks(persona: str):
    # URLs / handles / hashtags / emails
    if re.search(r"(https?://|www\.|@\w+|#\w+|\b\S+@\S+\b)", persona):
        return True

    # lots of digits (dates, scores, IDs, etc.)
    if len(re.findall(r"\d", persona)) >= 3:
        return True

    # quoted spans often indicate copying
    if re.search(r"['\"“”].{15,}['\"“”]", persona):
        return True

    # many Capitalized words mid-sentence (weak proxy for named entities)
    caps = re.findall(r"(?<!\.)\s+[A-Z][a-z]{2,}\b", persona)
    if len(caps) >= 3:
        return True

    return False

def sanitize_persona(persona: str):
    # remove DeepSeek thinking tags if any
    persona = persona.replace("<think>", "").replace("</think>", "").strip()

    # strip URLs/emails/handles
    persona = re.sub(r"(https?://\S+|www\.\S+|\b\S+@\S+\b|@\w+|#\w+)", "", persona)

    # remove long quoted fragments
    persona = re.sub(r"['\"“”].{15,}['\"“”]", "", persona)

    # collapse whitespace
    persona = re.sub(r"\s+", " ", persona).strip()
    return persona

def persona_is_ok(history: str, persona: str):
    p = persona.strip()
    if len(p.split()) < 45:   # too short to be meaningful
        return False
    if len(p.split()) > 140:  # too long; tends to leak
        return False
    if has_obvious_leaks(p):
        return False

    # block near-copy: no 5-gram overlap beyond tiny tolerance
    if overlap_ngram_score(history, p, n=5) > 0.02:
        return False

    # also block 6-gram overlap (even stricter)
    if overlap_ngram_score(history, p, n=6) > 0.01:
        return False

    return True


In [ ]:
import pandas as pd

persona_rows = []

for uid, hist in tqdm(user_histories.items(), total=len(user_histories)):
    # prompt: derived from history, but forced to be generic and non-specific
    base_prompt = f"""
You are given a user's interaction history.

Write a GENERIC persona derived from the history, but it must be non-identifying.

Hard constraints:
- 5 to 10 sentences.
- Do NOT mention any specific people, places, organizations, titles, events, quotes, or unique phrases.
- Do NOT include dates, numbers, URLs, usernames, hashtags, or IDs.
- Do NOT paraphrase a specific item; only summarize broad preferences and style.
- Use broad terms like "technology", "sports", "health", "entertainment", "business", etc. when needed.
- Output ONLY the persona text (no headings, no bullet points, no JSON, no analysis).

History:
{hist}

Persona:
""".strip()

    persona = None
    prompt = base_prompt

    # retry loop: regenerate if audit fails
    for attempt in range(3):
        raw = llm_generate(prompt, max_new_tokens=20)
        cand = sanitize_persona(raw)

        if persona_is_ok(hist, cand):
            persona = cand
            break

        # tighten prompt if it leaked / was too close
        prompt = base_prompt + "\n\nYour previous attempt was too specific. Rewrite more generic and abstract.\nPersona:"

    if persona is None:
        # last-resort fallback: still derived, but extremely generic
        persona = "The user prefers broad informational content and practical summaries. They engage with general technology and everyday topics. They tend to favor clear, concise responses. They show interest in learning-oriented material and explanatory writing. They avoid overly sensational or strongly opinionated content. They appreciate structured information and actionable takeaways."

    persona_rows.append({"userid": uid, "persona": persona})

persona_df = pd.DataFrame(persona_rows)
persona_df.to_csv("user_personas.csv", index=False)
persona_df.head()


In [ ]:
persona_df

In [46]:
persona_map = dict(zip(persona_df["userid"], persona_df["persona"]))
user_ids = list(persona_map.keys())


In [ ]:
import random

def clean_user_id(x, valid_ids):
    x = x.replace("<think>", "").replace("</think>", "").strip()
    for uid in valid_ids:
        if uid in x:
            return uid
    return "INVALID"

user_pred_results = []

for idx, row in tqdm(up_df.iterrows(), total=len(up_df)):
    true_uid = row["userid"]
    wrong_uid = random.choice([u for u in user_ids if u != true_uid])

    prompt = f"""
You must choose which user wrote the history.

Conversation:
{build_history(row)}

User A (ID: {true_uid})
{persona_map[true_uid]}

User B (ID: {wrong_uid})
{persona_map[wrong_uid]}

RULES:
- Output ONLY one of these two IDs: {true_uid} or {wrong_uid}
- Do NOT explain
- Do NOT use <think> tags
- Do NOT output anything else

Answer:
"""

    raw = llm_generate(prompt, max_new_tokens=16)
    pred = clean_user_id(raw, [true_uid, wrong_uid])

    user_pred_results.append({
        "userid": true_uid,
        "query_text_id": row["query_text_id"],
        "task": "next_user_prediction",
        "prediction": pred
    })


In [ ]:
pd.DataFrame(user_pred_results).to_csv("deepseek_user_predictions.csv", index=False)
print("Saved deepseek_user_predictions.csv")


In [ ]:
user_pred_results

In [ ]:
pred_df = pd.read_csv("deepseek_user_predictions.csv")

correct = (pred_df["userid"].astype(str) == pred_df["prediction"].astype(str))
accuracy = correct.mean() * 100

print(f"Speaker Identification Accuracy of DeepSeek-32B: {accuracy:.2f}%")
